# Day 14 — SQL for Data Science
### Cohort Analysis · Funnel Analysis · Retention · Date Arithmetic

## 1. Setup & Create Sample Business Data

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from datetime import datetime, timedelta

con = duckdb.connect()


def sql(query):
    return con.execute(query).df()


# Create realistic e-commerce dataset
np.random.seed(42)
n_users = 500

# Users table
users = pd.DataFrame(
    {
        "user_id": range(1, n_users + 1),
        "signup_date": pd.date_range("2024-01-01", periods=n_users, freq="12h").date,
        "country": np.random.choice(
            ["UK", "US", "DE", "FR"], n_users, p=[0.4, 0.3, 0.2, 0.1]
        ),
        "plan": np.random.choice(
            ["free", "basic", "premium"], n_users, p=[0.5, 0.3, 0.2]
        ),
    }
)

# Orders table
order_rows = []
for uid in range(1, n_users + 1):
    n_orders = np.random.poisson(2)
    signup = pd.Timestamp("2024-01-01") + pd.Timedelta(hours=12 * (uid - 1))
    for _ in range(n_orders):
        days_after = np.random.randint(0, 180)
        order_rows.append(
            {
                "order_id": len(order_rows) + 1,
                "user_id": uid,
                "order_date": (signup + pd.Timedelta(days=days_after)).date(),
                "amount": round(np.random.exponential(50), 2),
                "status": np.random.choice(
                    ["completed", "refunded", "pending"], p=[0.85, 0.1, 0.05]
                ),
            }
        )

orders = pd.DataFrame(order_rows)

# Events/funnel table
events_rows = []
stages = ["visited", "viewed_product", "added_to_cart", "checkout", "purchased"]
for uid in range(1, 201):
    signup = pd.Timestamp("2024-01-01") + pd.Timedelta(hours=12 * (uid - 1))
    max_stage = np.random.choice(range(5), p=[0.1, 0.2, 0.3, 0.2, 0.2])
    for i, stage in enumerate(stages[: max_stage + 1]):
        events_rows.append(
            {
                "user_id": uid,
                "event": stage,
                "event_date": (
                    signup + pd.Timedelta(days=np.random.randint(0, 30))
                ).date(),
            }
        )

events = pd.DataFrame(events_rows)

con.register("users", users)
con.register("orders", orders)
con.register("events", events)

print(f"Users:  {len(users)} rows")
print(f"Orders: {len(orders)} rows")
print(f"Events: {len(events)} rows")
print(f"\nUsers sample:")
print(users.head(3))
print(f"\nOrders sample:")
print(orders.head(3))
print("Ready! ✅")

Users:  500 rows
Orders: 1042 rows
Events: 626 rows

Users sample:
   user_id signup_date country   plan
0        1  2024-01-01      UK  basic
1        2  2024-01-01      FR  basic
2        3  2024-01-02      DE   free

Orders sample:
   order_id  user_id  order_date  amount     status
0         1        1  2024-03-23   65.88  completed
1         2        2  2024-03-29   52.02  completed
2         3        2  2024-04-18   55.42  completed
Ready! ✅


## 2. Cohort Analysis

In [ ]:
print("=" * 55)
print("         COHORT ANALYSIS")
print("=" * 55)
print(
    """
Cohort Analysis groups users by when they signed up
and tracks their behaviour over time.

WHY IT MATTERS:
  - Are newer users better than older users?
  - Is retention improving over time?
  - Which signup month produces best customers?
"""
)

# Monthly cohort — revenue per cohort
print("\n--- Q1: Revenue by signup month cohort ---")
print(
    sql(
        """
    WITH user_cohorts AS (
        SELECT 
            user_id,
            DATE_TRUNC('month', signup_date::DATE) as cohort_month
        FROM users
    ),
    cohort_orders AS (
        SELECT 
            uc.cohort_month,
            COUNT(DISTINCT uc.user_id) as cohort_size,
            COUNT(o.order_id) as total_orders,
            ROUND(SUM(o.amount), 2) as total_revenue,
            ROUND(AVG(o.amount), 2) as avg_order_value
        FROM user_cohorts uc
        LEFT JOIN orders o ON uc.user_id = o.user_id
            AND o.status = 'completed'
        GROUP BY uc.cohort_month
    )
    SELECT 
        cohort_month,
        cohort_size,
        total_orders,
        total_revenue,
        avg_order_value,
        ROUND(total_revenue / cohort_size, 2) as revenue_per_user
    FROM cohort_orders
    ORDER BY cohort_month
    LIMIT 8
"""
    )
)

# Orders per user by cohort
print("\n--- Q2: Orders per user by plan type ---")
print(
    sql(
        """
    SELECT 
        u.plan,
        COUNT(DISTINCT u.user_id) as users,
        COUNT(o.order_id) as total_orders,
        ROUND(COUNT(o.order_id)::FLOAT / COUNT(DISTINCT u.user_id), 2) as orders_per_user,
        ROUND(SUM(o.amount), 2) as total_revenue,
        ROUND(AVG(o.amount), 2) as avg_order_value
    FROM users u
    LEFT JOIN orders o ON u.user_id = o.user_id
        AND o.status = 'completed'
    GROUP BY u.plan
    ORDER BY orders_per_user DESC
"""
    )
)

         COHORT ANALYSIS

Cohort Analysis groups users by when they signed up
and tracks their behaviour over time.

WHY IT MATTERS:
  - Are newer users better than older users?
  - Is retention improving over time?
  - Which signup month produces best customers?


--- Q1: Revenue by signup month cohort ---
  cohort_month  cohort_size  total_orders  total_revenue  avg_order_value  \
0   2024-01-01           62           107        5962.90            55.73   
1   2024-02-01           58            96        4370.89            45.53   
2   2024-03-01           62           105        4501.09            42.87   
3   2024-04-01           60           103        5524.41            53.64   
4   2024-05-01           62           119        5916.91            49.72   
5   2024-06-01           60           124        6216.46            50.13   
6   2024-07-01           62           112        4076.16            36.39   
7   2024-08-01           62           100        4231.40            42.31  

## 3. Funnel Analysis

In [ ]:
print("=" * 55)
print("          FUNNEL ANALYSIS")
print("=" * 55)
print(
    """
Funnel Analysis tracks how users move through stages
of a process — and where they DROP OFF.

E-COMMERCE FUNNEL:
  Visited → Viewed Product → Added to Cart → Checkout → Purchased
  
WHERE do we lose most users?
WHICH stage needs most improvement?
"""
)

# Build the funnel
print("\n--- Q3: Full conversion funnel ---")
print(
    sql(
        """
    WITH stage_counts AS (
        SELECT 
            event,
            COUNT(DISTINCT user_id) as users,
            CASE event
                WHEN 'visited'         THEN 1
                WHEN 'viewed_product'  THEN 2
                WHEN 'added_to_cart'   THEN 3
                WHEN 'checkout'        THEN 4
                WHEN 'purchased'       THEN 5
            END as stage_order
        FROM events
        GROUP BY event
    ),
    funnel AS (
        SELECT *,
            MAX(users) OVER () as top_of_funnel
        FROM stage_counts
    )
    SELECT 
        stage_order,
        event as stage,
        users,
        ROUND(users::FLOAT / top_of_funnel * 100, 1) as pct_of_visitors,
        LAG(users) OVER (ORDER BY stage_order) as prev_stage_users,
        ROUND(
            users::FLOAT / LAG(users) OVER (ORDER BY stage_order) * 100
        , 1) as step_conversion_pct
    FROM funnel
    ORDER BY stage_order
"""
    )
)

# Funnel by country
print("\n--- Q4: Funnel conversion by country ---")
print(
    sql(
        """
    WITH purchased_users AS (
        SELECT DISTINCT e.user_id
        FROM events e
        WHERE e.event = 'purchased'
    ),
    visited_users AS (
        SELECT DISTINCT e.user_id
        FROM events e
        WHERE e.event = 'visited'
    )
    SELECT 
        u.country,
        COUNT(DISTINCT v.user_id) as visited,
        COUNT(DISTINCT p.user_id) as purchased,
        ROUND(COUNT(DISTINCT p.user_id)::FLOAT / 
              NULLIF(COUNT(DISTINCT v.user_id), 0) * 100, 1) as conversion_pct
    FROM users u
    LEFT JOIN visited_users v ON u.user_id = v.user_id
    LEFT JOIN purchased_users p ON u.user_id = p.user_id
    GROUP BY u.country
    ORDER BY conversion_pct DESC
"""
    )
)

          FUNNEL ANALYSIS

Funnel Analysis tracks how users move through stages
of a process — and where they DROP OFF.

E-COMMERCE FUNNEL:
  Visited → Viewed Product → Added to Cart → Checkout → Purchased

WHERE do we lose most users?
WHICH stage needs most improvement?


--- Q3: Full conversion funnel ---
   stage_order           stage  users  pct_of_visitors  prev_stage_users  \
0            1         visited    200            100.0              <NA>   
1            2  viewed_product    174             87.0               200   
2            3   added_to_cart    132             66.0               174   
3            4        checkout     79             39.5               132   
4            5       purchased     41             20.5                79   

   step_conversion_pct  
0                  NaN  
1            87.000000  
2            75.900002  
3            59.799999  
4            51.900002  

--- Q4: Funnel conversion by country ---
  country  visited  purchased  conversion_

## 4. Retention Analysis

In [ ]:
print("=" * 55)
print("         RETENTION ANALYSIS")
print("=" * 55)
print(
    """
Retention measures: of users who signed up in month X,
how many came back in month X+1, X+2, X+3?

HIGH retention = product is sticky = business is healthy
LOW retention = users sign up but never return = problem!
"""
)

# Monthly retention
print("\n--- Q5: Monthly user retention ---")
print(
    sql(
        """
    WITH first_order AS (
        SELECT 
            user_id,
            MIN(order_date::DATE) as first_order_date,
            DATE_TRUNC('month', MIN(order_date::DATE)) as first_order_month
        FROM orders
        WHERE status = 'completed'
        GROUP BY user_id
    ),
    subsequent_orders AS (
        SELECT 
            o.user_id,
            fo.first_order_month,
            DATE_TRUNC('month', o.order_date::DATE) as order_month,
            DATEDIFF('month', fo.first_order_month, 
                     DATE_TRUNC('month', o.order_date::DATE)) as months_since_first
        FROM orders o
        JOIN first_order fo ON o.user_id = fo.user_id
        WHERE o.status = 'completed'
    )
    SELECT 
        months_since_first,
        COUNT(DISTINCT user_id) as active_users,
        ROUND(COUNT(DISTINCT user_id)::FLOAT / 
              MAX(COUNT(DISTINCT user_id)) OVER () * 100, 1) as retention_pct
    FROM subsequent_orders
    GROUP BY months_since_first
    ORDER BY months_since_first
    LIMIT 7
"""
    )
)

# Day-1 retention by plan
print("\n--- Q6: Repeat purchase rate by plan ---")
print(
    sql(
        """
    WITH order_counts AS (
        SELECT 
            u.user_id,
            u.plan,
            COUNT(o.order_id) as total_orders
        FROM users u
        LEFT JOIN orders o ON u.user_id = o.user_id
            AND o.status = 'completed'
        GROUP BY u.user_id, u.plan
    )
    SELECT 
        plan,
        COUNT(*) as total_users,
        SUM(CASE WHEN total_orders >= 1 THEN 1 ELSE 0 END) as made_purchase,
        SUM(CASE WHEN total_orders >= 2 THEN 1 ELSE 0 END) as repeat_buyers,
        ROUND(SUM(CASE WHEN total_orders >= 1 THEN 1 ELSE 0 END)::FLOAT / 
              COUNT(*) * 100, 1) as purchase_rate_pct,
        ROUND(SUM(CASE WHEN total_orders >= 2 THEN 1 ELSE 0 END)::FLOAT / 
              NULLIF(SUM(CASE WHEN total_orders >= 1 THEN 1 ELSE 0 END), 0) * 100, 1) as repeat_rate_pct
    FROM order_counts
    GROUP BY plan
    ORDER BY repeat_rate_pct DESC
"""
    )
)

         RETENTION ANALYSIS

Retention measures: of users who signed up in month X,
how many came back in month X+1, X+2, X+3?

HIGH retention = product is sticky = business is healthy
LOW retention = users sign up but never return = problem!


--- Q5: Monthly user retention ---
   months_since_first  active_users  retention_pct
0                   0           414     100.000000
1                   1            96      23.200001
2                   2            90      21.700001
3                   3            82      19.799999
4                   4            49      11.800000
5                   5            33       8.000000
6                   6             4       1.000000

--- Q6: Repeat purchase rate by plan ---
      plan  total_users  made_purchase  repeat_buyers  purchase_rate_pct  \
0    basic          145          128.0           89.0          88.300003   
1  premium           93           75.0           49.0          80.599998   
2     free          262          211.0    

## 5. Date Arithmetic

In [ ]:
print("=" * 55)
print("          DATE ARITHMETIC")
print("=" * 55)

# Days between signup and first order
print("\n--- Q7: Days from signup to first order ---")
print(
    sql(
        """
    WITH first_orders AS (
        SELECT 
            o.user_id,
            MIN(o.order_date::DATE) as first_order_date
        FROM orders o
        WHERE o.status = 'completed'
        GROUP BY o.user_id
    )
    SELECT 
        DATEDIFF('day', u.signup_date::DATE, fo.first_order_date) as days_to_first_order,
        COUNT(*) as users,
        ROUND(AVG(DATEDIFF('day', u.signup_date::DATE, fo.first_order_date)), 1) as avg_days
    FROM users u
    JOIN first_orders fo ON u.user_id = fo.user_id
    GROUP BY days_to_first_order
    ORDER BY days_to_first_order
    LIMIT 8
"""
    )
)

# Revenue by day of week
print("\n--- Q8: Revenue by day of week ---")
print(
    sql(
        """
    SELECT 
        DAYNAME(order_date::DATE) as day_name,
        DAYOFWEEK(order_date::DATE) as day_num,
        COUNT(*) as orders,
        ROUND(SUM(amount), 2) as revenue,
        ROUND(AVG(amount), 2) as avg_order
    FROM orders
    WHERE status = 'completed'
    GROUP BY day_name, day_num
    ORDER BY day_num
"""
    )
)

# Revenue by month
print("\n--- Q9: Monthly revenue trend ---")
print(
    sql(
        """
    SELECT 
        DATE_TRUNC('month', order_date::DATE) as month,
        COUNT(*) as orders,
        COUNT(DISTINCT user_id) as unique_buyers,
        ROUND(SUM(amount), 2) as revenue,
        ROUND(AVG(amount), 2) as avg_order,
        ROUND(SUM(amount) - LAG(SUM(amount)) OVER (ORDER BY DATE_TRUNC('month', order_date::DATE)), 2) as mom_change
    FROM orders
    WHERE status = 'completed'
    GROUP BY month
    ORDER BY month
    LIMIT 8
"""
    )
)

          DATE ARITHMETIC

--- Q7: Days from signup to first order ---
   days_to_first_order  users  avg_days
0                    0      3       0.0
1                    1      6       1.0
2                    2      3       2.0
3                    3      4       3.0
4                    4      1       4.0
5                    5      5       5.0
6                    6      3       6.0
7                    7      8       7.0

--- Q8: Revenue by day of week ---
    day_name  day_num  orders  revenue  avg_order
0     Sunday        0     126  4901.50      38.90
1     Monday        1     116  5475.45      47.20
2    Tuesday        2     131  6287.04      47.99
3  Wednesday        3     128  5905.49      46.14
4   Thursday        4     119  4932.51      41.45
5     Friday        5     135  6996.25      51.82
6   Saturday        6     129  6976.63      54.08

--- Q9: Monthly revenue trend ---
       month  orders  unique_buyers  revenue  avg_order  mom_change
0 2024-01-01       9          

## 6. Key Takeaways — Day 14 🎯

### Cohort Analysis
- Groups users by signup period and tracks behaviour over time
- Answers: "Are newer users better than older ones?"
- June cohort: £103.61 revenue/user — best performing cohort
- Basic plan: 2.01 orders/user — most active buyers

### Funnel Analysis
- Tracks user progression through stages — finds drop-off points
- Our funnel: 200 visitors → 41 purchased = 20.5% conversion
- Biggest drop: Checkout → Purchase (only 51.9% completed!)
- France: 0% conversion — critical issue to investigate!
- Fix the checkout page first — highest impact improvement

### Retention Analysis
- Month 1 retention: only 23.2% — lost 77% after first order!
- Classic leaky bucket — acquisition working, retention broken
- Basic plan: 69.5% repeat rate — best customer segment
- Healthy benchmark: Month 1 retention should be 40%+

### Date Arithmetic Functions
| Function | Purpose | Example |
|---|---|---|
| DATE_TRUNC('month', date) | Round down to month | Group by month |
| DATEDIFF('day', date1, date2) | Days between dates | Days to first order |
| DAYNAME(date) | Day name string | 'Monday', 'Friday' |
| DAYOFWEEK(date) | Day number (0-6) | 0=Sunday, 6=Saturday |
| LAG() OVER (ORDER BY month) | Previous month value | Month-over-month change |

### Key Business Insights
- Friday & Saturday: highest revenue days
- July: best revenue month (+£3,743 vs June)
- 3 users bought on signup day — highly engaged segment
- Basic plan users outperform premium users in engagement!

### SQL Patterns Learned
- Cohort: DATE_TRUNC + LEFT JOIN + GROUP BY cohort
- Funnel: COUNT DISTINCT per stage + LAG for step conversion
- Retention: DATEDIFF between first order and subsequent orders
- Date trends: DATE_TRUNC + LAG for month-over-month change